In [1]:
import getpass
import os


def set_api_key(key_name: str) -> None:
    """
    Securely set an environment variable if it doesn't already exist.
    Prompts the user for input using a password-style hidden input.
    
    Args:
        key_name (str): Name of the environment variable to set (e.g., "OPENAI_API_KEY")
    """
    if not os.environ.get(key_name):
        os.environ[key_name] = getpass.getpass(f"{key_name}: ")

# Example usage:
set_api_key("OPENAI_API_KEY")
# set_api_key("ANTHROPIC_API_KEY")

In [2]:
os.environ["LANGCHAIN_TRACING_V2"] = "false"
# os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
set_api_key("LANGCHAIN_API_KEY")

In [3]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

In [4]:
len(docs)

64

In [5]:
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_openai import OpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from sqlalchemy import true

  # LM Studio uses OpenAI-compatible API format
generator_llm = ChatOpenAI(
      base_url="http://192.168.1.157:1234/v1",  # Replace with your server 
      api_key="lmstudio",  # LM Studio doesn't require real API key
      model="openai/gpt-oss-120b",  # Optional: specify model name
      temperature=0.7,
      verbose=True 
  )

try:
  response = generator_llm.invoke("Test message")
  print(response)
except Exception as e:
  print(f"Error details: {e}")
  
  # Wrap for RAGAS
ragas_llm = LangchainLLMWrapper(generator_llm)
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

  # Use in RAGAS evaluation
#   results = evaluate(
#       dataset=your_dataset,
#       metrics=[faithfulness, answer_relevancy],
#       llm=ragas_llm,
#       embeddings=ragas_embeddings
#   )

content='Message received! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 69, 'total_tokens': 97, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'openai/gpt-oss-120b', 'id': 'chatcmpl-0faqojl8ikvvhg7qssrcb7u', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None} id='run--77d9f218-4280-49f8-9e39-4c6dfb38a0eb-0' usage_metadata={'input_tokens': 69, 'output_tokens': 28, 'total_tokens': 97, 'input_token_details': {}, 'output_token_details': {}}


In [6]:
from ragas.testset.graph import KnowledgeGraph
kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

In [7]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

In [8]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = ragas_llm
embedding_model = ragas_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

Property 'summary' already exists in node 'b214f2'. Skipping!
Property 'summary' already exists in node '1124c8'. Skipping!
Property 'summary' already exists in node '802291'. Skipping!
Property 'summary' already exists in node 'a2d5be'. Skipping!
Property 'summary' already exists in node '1bbb33'. Skipping!
Property 'summary' already exists in node 'c0ea8d'. Skipping!
Property 'summary' already exists in node '889f15'. Skipping!
Property 'summary' already exists in node '3afe21'. Skipping!
Property 'summary' already exists in node '83a9ad'. Skipping!
Property 'summary' already exists in node '3a3d1c'. Skipping!
Property 'summary' already exists in node '4a1c2b'. Skipping!
Property 'summary' already exists in node 'eb5f8f'. Skipping!
Property 'summary' already exists in node '16d391'. Skipping!
Property 'summary' already exists in node '99d760'. Skipping!
Property 'summary' already exists in node 'eaea6a'. Skipping!
Property 'summary' already exists in node '397276'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/44 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'b214f2'. Skipping!
Property 'summary_embedding' already exists in node '1124c8'. Skipping!
Property 'summary_embedding' already exists in node '802291'. Skipping!
Property 'summary_embedding' already exists in node 'a2d5be'. Skipping!
Property 'summary_embedding' already exists in node '1bbb33'. Skipping!
Property 'summary_embedding' already exists in node 'c0ea8d'. Skipping!
Property 'summary_embedding' already exists in node '889f15'. Skipping!
Property 'summary_embedding' already exists in node '3afe21'. Skipping!
Property 'summary_embedding' already exists in node '83a9ad'. Skipping!
Property 'summary_embedding' already exists in node '3a3d1c'. Skipping!
Property 'summary_embedding' already exists in node '4a1c2b'. Skipping!
Property 'summary_embedding' already exists in node 'eb5f8f'. Skipping!
Property 'summary_embedding' already exists in node '16d391'. Skipping!
Property 'summary_embedding' already exists in node '99d760'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 85, relationships: 782)

In [ ]:
kg.save("graph_created_oss120.json")
usecase_data_kg=KnowledgeGraph.load("graph_created_oss120.json")
usecase_data_kg

In [ ]:
# from ragas.testset import TestsetGenerator
# from ragas.testset.transforms import default_transforms

# generator = TestsetGenerator(llm=generator_llm, embedding_model=ragas_embeddings)
# dataset = generator.generate_with_langchain_docs(documents=docs, 
#     testset_size=10,
#     transforms=default_transforms,
#     transforms_llm=ragas_llm,
#     transforms_embedding_model=ragas_embeddings,
#     with_debugging_logs=True
#     )

In [ ]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=ragas_llm, embedding_model=ragas_embeddings)

In [ ]:
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

In [ ]:
generator_llm.invoke("What is the capital of Germany")

In [ ]:
generator_llm.invoke("Explain quantum mechanics for a six year old.")

In [ ]:
dataset2 = generator.generate_with_langchain_docs(docs, testset_size=10)

In [ ]:
dataset.to_pandas()

In [ ]:
  # LM Studio uses OpenAI-compatible API format
fast_generator_llm = ChatOpenAI(
      base_url="http://192.168.1.157:1234/v1",  # Replace with your server 
      api_key="lmstudio",  # LM Studio doesn't require real API key
      model="openai/gpt-oss-20b",  # Optional: specify model name
      temperature=0.7,
      verbose=True 
  )

try:
  response = fast_generator_llm.invoke("Test message")
  print(response)
except Exception as e:
  print(f"Error details: {e}")

In [ ]:
fast_ragas_llm = LangchainLLMWrapper(fast_generator_llm)

In [ ]:
fast_generator = TestsetGenerator(llm=fast_ragas_llm, embedding_model=ragas_embeddings)
dataset = fast_generator.generate_with_langchain_docs(docs, testset_size=10)

In [ ]:
dataset.to_pandas()


In [ ]:
from ragas.testset.graph import KnowledgeGraph
kg_fast = KnowledgeGraph()
kg_fast

In [ ]:

for doc in docs:
    kg_fast.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg_fast

In [ ]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = fast_ragas_llm
embedding_model = ragas_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg_fast, default_transforms)
kg_fast

In [ ]:
kg_fast.save("graph_created_oss20b.json")
fast_usecase_data_kg=KnowledgeGraph.load("graph_created_oss20b.json")
fast_usecase_data_kg

In [ ]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=fast_ragas_llm, embedding_model=ragas_embeddings, knowledge_graph=fast_usecase_data_kg)

In [ ]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=fast_ragas_llm), 0.1),
        (MultiHopAbstractQuerySynthesizer(llm=fast_ragas_llm), 0.45),
        (MultiHopSpecificQuerySynthesizer(llm=fast_ragas_llm), 0.45),
]

In [ ]:
big_testset = generator.generate(testset_size=10, query_distribution=query_distribution)
big_testset.to_pandas()